# RAG System - Colab Experiment
This notebook allows you to run the RAG system on Google Colab and persist the FAISS index to Google Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Dependencies
!pip install pypdf sentence-transformers faiss-cpu ctransformers

In [ ]:
# 3. Clone/Setup Path (Assuming you uploaded the code or cloned it)
import sys
import os

# ADJUST THIS PATH to where your folder is in Drive
project_path = '/content/drive/MyDrive/rag-system'

if project_path not in sys.path:
    sys.path.append(project_path)

# Verify imports
try:
    from ingestion.load_docs import DocumentLoader
    print("Imports successful!")
except ImportError:
    print("Could not import modules. Check project_path.")

In [ ]:
# 4. Configuration
import os

# We will store embeddings in Drive so they persist after runtime reset
DATA_DIR = os.path.join(project_path, "data")
RAW_DOCS_DIR = os.path.join(DATA_DIR, "raw_docs")
EMBEDDING_DIR = os.path.join(project_path, "embeddings") # Persisted in Drive

print(f"Raw Docs: {RAW_DOCS_DIR}")
print(f"Embeddings: {EMBEDDING_DIR}")

In [ ]:
# 5. Run Cleanup & Ingestion
import shutil
from ingestion.load_docs import DocumentLoader
from ingestion.chunk_docs import DocumentChunker
from ingestion.embed_docs import EmebeddingGenerator
from retrieval.vector_store import VectorStore

def run_ingestion():
    # Load
    print("Loading docs...")
    loader = DocumentLoader(RAW_DOCS_DIR)
    docs = loader.load_documents()
    if not docs:
        print("No docs found!")
        return

    # Chunk
    print("Chunking...")
    chunker = DocumentChunker()
    chunks = chunker.chunk_documents(docs)

    # Embed
    print("Embedding...")
    gen = EmebeddingGenerator()
    embeddings = gen.generate_embeddings(chunks)
    gen.save_embeddings(embeddings, chunks, EMBEDDING_DIR)

    # Index
    print("Indexing...")
    index_path = os.path.join(EMBEDDING_DIR, "faiss_index.index")
    metadata_path = os.path.join(EMBEDDING_DIR, "metadata.pkl")
    
    store = VectorStore(index_path, metadata_path)
    store.build_index(embeddings, chunks)
    store.save_index()
    print("Done!")

# Uncommon the line below to run ingestion
# run_ingestion()

In [ ]:
# 6. Retrieval & QA
from retrieval.search import SearchEngine
from generation.prompt import PromptEngineering
from generation.llm import LLMService

# Initialize Engine with Drive paths
# We strictly pass the base_dir to help it find the persisted index
search_engine = SearchEngine(base_dir=project_path)
llm_service = LLMService()

query = "What is the main contribution of this paper?"
chunks = search_engine.search(query, k=3)

if chunks:
    prompt = PromptEngineering.build_prompt(query, chunks)
    response = llm_service.generate_response(prompt)
    print(f"\nAnswer: {response['answer']}\n")
    print(f"Sources: {[c['source'] for c in chunks]}")
else:
    print("No relevant chunks found.")